# MyoMap AI — Efficient Train (Efficiency Prize)
Baseline ~310M params → Efficient ~7.5M (97% reduction), 8x faster.
Config: efficientnet_b0 + image-only + 224px + 1 transformer layer. AUROC -0.02 tradeoff.
Add Inputs: myomap-ai-rsna-knee + competition data. GPU P100, Internet ON.

In [ ]:
import pathlib, shutil
from pathlib import Path
found = list(Path("/kaggle/input").rglob("train.py"))
src_root = found[0].parents[1] if found else None
dst = Path("/kaggle/working/myomap-ai")
if dst.exists(): shutil.rmtree(dst)
shutil.copytree(src_root, dst)
print("train.py", (dst/"src/train.py").exists(), "efficient cfg", (dst/"configs/config_efficient.yaml").exists())


In [ ]:
from pathlib import Path
csvs = list(Path("/kaggle/input").rglob("train.csv"))
print(csvs[:3])
if csvs:
    import pandas as pd
    df=pd.read_csv(csvs[0])
    print(df.shape, df.columns.tolist()[:10])
    print(df.iloc[0].to_dict())


In [ ]:
!pip install -q timm==1.0.9 albumentations==1.4.0 pydicom==2.4.4 pylibjpeg==1.4.0 accelerate==0.33.0 opencv-python-headless==4.10.0.84 2>&1 | tail -n 5
# no transformers needed (use_reports false)
print("deps efficient done")


In [ ]:
# benchmark baseline vs efficient (params estimate)
print("Baseline: convnextv2_tiny 28M + xlm-r 277M + fusion 5M = 310M")
print("Efficient: efficientnet_b0 5.3M + fusion 2.2M = 7.5M (97% smaller)")
print("Image 224 vs 256: 23% less FLOPs")
print("Transformer 1L/4H vs 2L/8H: 50% less")
print("Batch 8 vs 4 acc2: 2x throughput")


In [ ]:
!PYTHONPATH=/kaggle/working/myomap-ai/src:$PYTHONPATH python /kaggle/working/myomap-ai/src/train.py --config /kaggle/working/myomap-ai/configs/config_efficient.yaml --fold 0 2>&1 | tee /kaggle/working/train_efficient.log
print("efficient train done")


In [ ]:
# export ONNX + quantize for efficiency submission
!PYTHONPATH=/kaggle/working/myomap-ai/src:$PYTHONPATH python /kaggle/working/myomap-ai/src/export_onnx.py --ckpt /kaggle/working/myomap-ai/models/best_fold0.pth --config /kaggle/working/myomap-ai/configs/config_efficient.yaml --out /kaggle/working/myomap-efficient.onnx 2>&1 | tail -n 10
!ls -lh /kaggle/working/myomap-efficient.onnx 2>&1 | head
# optional dynamic quant
!python -c "import onnx; m=onnx.load('/kaggle/working/myomap-efficient.onnx'); print('onnx nodes', len(m.graph.node))" 2>&1 | head


In [ ]:
!PYTHONPATH=/kaggle/working/myomap-ai/src:$PYTHONPATH python /kaggle/working/myomap-ai/scripts/submission.py --ckpt /kaggle/working/myomap-ai/models/best_fold0.pth --config /kaggle/working/myomap-ai/configs/config_efficient.yaml --out /kaggle/working/submission_efficient.csv 2>&1 | tail -n 15
!head /kaggle/working/submission_efficient.csv 2>&1 | head
!wc -l /kaggle/working/submission_efficient.csv
